# Guardian AI — Training Platform on Google Colab

This notebook runs the Sprint 17 training platform end to end on a Colab GPU (or CPU) runtime:

**Dataset → Train → Evaluate → Export ONNX → Validate → Copy to Drive**

Rules that still apply here:
- Datasets are loaded **only** through the Dataset Registry (Sprint 7). No loose folders.
- Every run produces an `experiment.json` — no anonymous models.
- The exported ONNX is validated (structure + torch/onnxruntime parity) before you may use it.
- Promotion to the Guardian Model Zoo is **never** done from Colab — it requires manual approval via `python -m guardian_ai.train promote` on a reviewed machine.

> Until the architecture review approves real training, use the synthetic `dummy-detection` dataset and the `tiny-ssd` smoke family, exactly as below.

## 1. Mount Google Drive

Drive holds your copy of the repository (or you can `git clone` instead) and receives the final ONNX artifact.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_OUT = "/content/drive/MyDrive/guardian-ai-models"
!mkdir -p {DRIVE_OUT}

## 2. Get the code and install dependencies

Clone the repository and install the `ai/` package. PyTorch/onnxruntime versions come from `ai/pyproject.toml` — do not pin different ones here.

In [ ]:
!git clone https://github.com/Xaliljon/guardian-ai.git /content/guardian-ai
%cd /content/guardian-ai
!pip install -q -e ./ai

## 3. Prepare the dataset (registry-only)

For smoke runs, generate and publish the synthetic dataset. For a real dataset (after architecture review), copy the registered dataset bundle from Drive into the registry path instead — training will refuse anything that did not pass the registry's quality and privacy gates.

In [ ]:
!python ai/training/scripts/make_dummy_dataset.py \
    --registry ai/training/datasets/registry --version 1.0.0

## 4. Train

One command, one YAML. The config is the complete recipe (model family, dataset coordinates, optimizer, seed) so the run is reproducible anywhere. The run directory under `ai/training/runs/` gets `experiment.json`, checkpoints and reports.

In [ ]:
!python -m guardian_ai.train train --config ai/training/configs/smoke.yaml

# Pick up the run directory of the newest experiment:
import pathlib

RUN = sorted(pathlib.Path("ai/training/runs").iterdir())[-1]
print("run:", RUN)

If Colab disconnects mid-training, reconnect and continue from the last checkpoint — optimizer, scheduler and early-stopping state are all restored:

```
!python -m guardian_ai.train resume --run {RUN}
```

## 5. Evaluate

Computes the full mandated metric set on the test split — precision, recall, F1, mAP@50, mAP@50-95, confusion matrix, absolute FP/FN — into `reports/evaluation.json`, then renders the visual reports (PNG + PDF).

In [ ]:
!python -m guardian_ai.train evaluate --run {RUN}
!python -m guardian_ai.train report --run {RUN}

from IPython.display import Image as ShowImage
from IPython.display import display

display(ShowImage(filename=f"{RUN}/reports/confusion_matrix.png"))
display(ShowImage(filename=f"{RUN}/reports/loss_curve.png"))

## 6. Export to ONNX (validated)

Export runs the ONNX structural checker **and** a torch-vs-onnxruntime parity inference; a diverging artifact is rejected, never written into a manifest. The manifest (`export/manifest.json`) is generated automatically with the SHA256, labels, dataset/taxonomy versions, git commit and license — never edit it by hand. The Guardian compatibility check (batch=1, float32, named tensors, allowed license) runs in the same step.

In [ ]:
!python -m guardian_ai.train export --run {RUN} --version 0.0.1
!python -m guardian_ai.train benchmark --run {RUN}

## 7. Copy the ONNX model to Drive

The artifact pair (model + auto-generated manifest) goes to Drive together. From there, a reviewer runs `compare` against the current baseline and — only with explicit human approval — `promote` into the Guardian Model Zoo. Nothing in this notebook can put a model on an Edge Box.

In [ ]:
!cp {RUN}/export/model.onnx {RUN}/export/manifest.json {DRIVE_OUT}/
!ls -la {DRIVE_OUT}
print("Done. Hand the artifact to review — promotion stays manual.")